# Preprocessing & Feature Engineering

This notebook prepares the diabetic dataset for machine learning by:
- Inspecting data quality
- Checking missing values
- Dropping low-value columns
- Handling '?' placeholders
- Encoding categorical variables
- Scaling numeric features
- Creating train/test splits


In [ ]:
# Set working directory and load dataset

import os
import pandas as pd
import numpy as np

In [ ]:
# Set working directory to project root
os.chdir(r"d:\NHS-ML-Project\hospital-readmission-ml-and-nhs-ae-dashboard")

In [ ]:
# Load the raw diabetic dataset
df = pd.read_csv("data/ml_raw/diabetic_data.csv")

# Preview the first few rows
df.head()


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Check Dataset Structure

We inspect the shape and data types to understand the dataset before
performing any cleaning or preprocessing.


In [ ]:
#  Check shape and data types

# Number of rows and columns
df.shape


# Data types and non-null counts
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

## Check Missing Values

We calculate missing values for each column to identify features with
high missingness that may need to be dropped or imputed.


In [ ]:
#  Check missing values per column

# Count missing values
missing_counts = df.isna().sum()

# Percentage of missing values
missing_percent = (missing_counts / len(df)) * 100

# Combine into a table
missing_table = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_percent': missing_percent
})

# Sort by highest missing percentage
missing_table.sort_values(by='missing_percent', ascending=False)


,missing_count,missing_percent
max_glu_serum,96420,94.746772
A1Cresult,84748,83.277322
race,0,0.000000
gender,0,0.000000
age,0,0.000000
weight,0,0.000000
admission_type_id,0,0.000000
discharge_disposition_id,0,0.000000
admission_source_id,0,0.000000
time_in_hospital,0,0.000000


## Identify '?' as Missing Values
The dataset uses '?' as a placeholder for missing values in several
categorical columns. We count how often '?' appears in each column.


In [ ]:
#  Count '?' values in each column

question_mark_counts = (df == "?").sum()

# Show only columns that contain '?'
question_mark_counts[question_mark_counts > 0].sort_values(ascending=False)


weight               98569
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
dtype: int64

##  Drop Columns with Extremely High Missing Values

Some columns in the dataset contain extremely high proportions of missing values 
or placeholder values ('?'). These columns do not provide useful information for 
modelling and can negatively impact data quality. 

We remove the following columns (only if they exist in the dataset):
- max_glu_serum
- A1Cresult
- examide
- citoglipton

This helps simplify the dataset and improves the reliability of downstream 
preprocessing and modelling steps.


In [ ]:
# Drop high-missing columns ONLY if they exist

cols_to_drop = ['max_glu_serum', 'A1Cresult', 'examide', 'citoglipton']

# Keep only columns that are present in the dataframe
cols_to_drop = [col for col in cols_to_drop if col in df.columns]

df = df.drop(columns=cols_to_drop)

df.shape


(101766, 46)

##  Replace '?' With NaN

The dataset uses the character '?' to represent missing values instead of 
standard NaN. To ensure proper handling of missing data, we replace all '?' 
entries with actual NaN values. This allows pandas and scikit-learn to treat 
them correctly during preprocessing.


In [ ]:
# ---------------------------------------------
#  Replace '?' with NaN
# ---------------------------------------------

df = df.replace("?", np.nan)

# Check missing values again
df.isna().sum().sort_values(ascending=False).head(20)


weight                      98569
medical_specialty           49949
payer_code                  40256
race                         2273
diag_3                       1423
diag_2                        358
diag_1                         21
age                             0
admission_type_id               0
discharge_disposition_id        0
time_in_hospital                0
gender                          0
patient_nbr                     0
encounter_id                    0
num_procedures                  0
num_lab_procedures              0
admission_source_id             0
num_medications                 0
number_inpatient                0
number_emergency                0
dtype: int64

##  Create Binary Target Column (readmitted_30)

The dataset contains the column `readmitted`, not `readmitted_30`.  
We convert it into a binary target:

- <30 → 1  
- NO  → 0  
- >30 → 0  

This creates the correct target variable for modelling.


In [ ]:
# ---------------------------------------------
# Create binary target column
# ---------------------------------------------

df['readmitted_30'] = df['readmitted'].map({
    '<30': 1,
    'NO': 0,
    '>30': 0
})

# Drop the original readmitted column
df = df.drop(columns=['readmitted'])

# Ensure target is numeric so it does NOT get one-hot encoded
df['readmitted_30'] = df['readmitted_30'].astype(int)

# Confirm
df['readmitted_30'].value_counts()


readmitted_30
0    90409
1    11357
Name: count, dtype: int64

## Fill Missing Values

After converting '?' to NaN, some categorical columns still contain missing values.
We fill these with 'Unknown' to avoid issues during encoding and model training.


In [ ]:
#  Fill missing values in categorical columns

# Identify categorical columns
cat_cols = df.select_dtypes(include='object').columns

# Fill missing values with 'Unknown'
df[cat_cols] = df[cat_cols].fillna("Unknown")

# Confirm no missing values remain in categorical columns
df[cat_cols].isna().sum().sum()


np.float64(0.0)

## Convert Categorical Columns to Category Type

Converting object columns to 'category' reduces memory usage and speeds up
encoding during preprocessing.


In [ ]:
# Convert object columns to 'category' dtype

for col in cat_cols:
    df[col] = df[col].astype('category')

# Check updated data types
df.dtypes.head(20)


encounter_id                   int64
patient_nbr                    int64
race                        category
gender                      category
age                         category
weight                      category
admission_type_id              int64
discharge_disposition_id       int64
admission_source_id            int64
time_in_hospital               int64
payer_code                  category
medical_specialty           category
num_lab_procedures             int64
num_procedures                 int64
num_medications                int64
number_outpatient              int64
number_emergency               int64
number_inpatient               int64
diag_1                      category
diag_2                      category
dtype: object

##  Convert Categorical Columns to Category Type

Converting object columns to category dtype reduces memory usage and speeds up 
encoding in the next step. This is especially useful for large healthcare 
datasets with many categorical variables.


In [ ]:
# ---------------------------------------------
#  Convert object columns to category dtype
# ---------------------------------------------

for col in cat_cols:
    df[col] = df[col].astype('category')


In [ ]:
df.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'metformin', 'repaglinide',
       'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide',
       'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
       'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
       'insulin', 'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted_30'],
      dtype='object')

## One-Hot Encode Categorical Variables

Machine learning models require numerical inputs. We convert categorical 
variables into numeric dummy variables using one-hot encoding. We drop the 
first category to avoid multicollinearity.


In [ ]:
# ---------------------------------------------
#  One-hot encode categorical variables
# ---------------------------------------------

df_encoded = pd.get_dummies(df, drop_first=True)

# Check encoded shape
df_encoded.shape


(101766, 2431)

In [ ]:
df.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'metformin', 'repaglinide',
       'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide',
       'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
       'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
       'insulin', 'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted_30'],
      dtype='object')

## Scale Numeric Features

We scale only the numeric feature columns (excluding the target) using 
StandardScaler. This ensures all numeric features are on a similar scale, 
which improves model performance for algorithms like Logistic Regression.


In [ ]:
# ---------------------------------------------
# Scale numeric features
# ---------------------------------------------

from sklearn.preprocessing import StandardScaler

# Identify numeric columns EXCEPT the target
num_cols = df_encoded.drop(columns=['readmitted_30']).select_dtypes(include=['int64','float64']).columns

# Initialize scaler
scaler = StandardScaler()

# Scale numeric columns
df_encoded[num_cols] = scaler.fit_transform(df_encoded[num_cols])


## Train/Test Split

We split the dataset into training and testing sets. Stratification ensures
the target variable has the same distribution in both sets.


In [ ]:
# Train/test split

from sklearn.model_selection import train_test_split

# Separate features and target
X = df_encoded.drop('readmitted_30', axis=1)
y = df_encoded['readmitted_30']

# Split into train and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Show shapes
X_train.shape, X_test.shape


((81412, 2430), (20354, 2430))

## Train Logistic Regression Model

We begin model training with Logistic Regression, a strong baseline model for
binary classification. It is fast, interpretable, and works well with scaled
features. We train the model using the training set and evaluate it on the
test set using accuracy, precision, recall, F1-score, and the confusion matrix.


In [ ]:
# ---------------------------------------------
# Logistic Regression Model
# ---------------------------------------------

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Initialize model
log_reg = LogisticRegression(max_iter=500)

# Train model
log_reg.fit(X_train, y_train)

# Predict on test set
y_pred_lr = log_reg.predict(X_test)

# Evaluation
print("Logistic Regression Performance:\n")
print(classification_report(y_test, y_pred_lr))
print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_lr))


Logistic Regression Performance:

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     18083
           1       0.47      0.02      0.03      2271

    accuracy                           0.89     20354
   macro avg       0.68      0.51      0.49     20354
weighted avg       0.84      0.89      0.84     20354

Confusion Matrix:

[[18037    46]
 [ 2230    41]]


## Random Forest Model

Random Forest is an ensemble of decision trees. It handles non-linear patterns 
and interactions between features effectively. We train the model and evaluate 
its performance on the test set.


In [ ]:
# ---------------------------------------------
#  Random Forest Model
# ---------------------------------------------

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Initialize model
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

# Train model
rf.fit(X_train, y_train)

# Predict
y_pred_rf = rf.predict(X_test)

# Evaluation
print("Random Forest Performance:\n")
print(classification_report(y_test, y_pred_rf))
print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_rf))


Random Forest Performance:

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     18083
           1       0.65      0.01      0.01      2271

    accuracy                           0.89     20354
   macro avg       0.77      0.50      0.48     20354
weighted avg       0.86      0.89      0.84     20354

Confusion Matrix:

[[18075     8]
 [ 2256    15]]


##   XGBoost Model

XGBoost is a powerful gradient boosting algorithm that often achieves 
state-of-the-art performance on structured healthcare datasets. We train the 
model and evaluate its performance.


In [ ]:
!pip install xgboost


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   -- ------------------------------------- 6.3/101.7 MB 36.3 MB/s eta 0:00:03
   ----- ---------------------------------- 13.9/101.7 MB 36.3 MB/s eta 0:00:03
   -------- ------------------------------- 22.0/101.7 MB 36.7 MB/s eta 0:00:03
   ----------- ---------------------------- 29.9/101.7 MB 37.3 MB/s eta 0:00:02
   --------------- ------------------------ 38.3/101.7 MB 37.5 MB/s eta 0:00:02
   ------------------ --------------------- 46.7/101.7 MB 37.8 MB/s eta 0:00:02
   --------------------- ------------------ 54.3/101.7 MB 37.7 MB/s eta 0:00:02
   ------------------------ --------------- 62.1/101.7 MB 37.8 MB/s eta 0:00:02
   --------------------------- ------------ 70.5/101.7 MB 37.9 MB/s eta 0:00:01
   ------------------------------ --------- 78.4/101.7 MB 38.0 MB/s eta 0:00:01
   ---------------------------------- ----- 86.8/101


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


##  Clean Column Names for XGBoost Compatibility

XGBoost does not allow feature names that contain special characters such as 
`[`, `]`, `<`, or `>`. After one‑hot encoding, some column names may include 
these characters (for example: `age_[70-80)` or `diag_1_<250`). 

To prevent XGBoost from throwing the error:

**ValueError: feature_names must be string, and may not contain [, ] or <**

we clean the column names by replacing these characters with safe alternatives. 
This step must be done *after* one‑hot encoding and *before* training the 
XGBoost model.


In [ ]:
# Clean column names for XGBoost compatibility
df_encoded.columns = (
    df_encoded.columns
    .str.replace('[', '(', regex=False)
    .str.replace(']', ')', regex=False)
    .str.replace('<', 'lt_', regex=False)
    .str.replace('>', 'gt_', regex=False)
)


## Train/Test Split

We separate the dataset into features (X) and the target variable (y).  
The target column is `readmitted_30`, which indicates whether a patient was 
readmitted within 30 days.

We then split the data into training and testing sets using an 80/20 split.  
Stratification ensures that the proportion of readmitted vs. not-readmitted 
patients remains consistent in both sets, which is important for imbalanced 
healthcare datasets.


In [ ]:
# ---------------------------------------------
# Create X, y and Train/Test Split
# ---------------------------------------------

from sklearn.model_selection import train_test_split

# Features and target
X = df_encoded.drop('readmitted_30', axis=1)
y = df_encoded['readmitted_30']

# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape


##  Retrain All Models

After cleaning column names and recreating the train/test split, all previous
models become invalid. We retrain Logistic Regression, Random Forest, and
XGBoost on the new X_train and y_train.


In [ ]:
# ---------------------------------------------
#Retrain Logistic Regression, Random Forest, XGBoost
# ---------------------------------------------

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Logistic Regression
log_reg = LogisticRegression(max_iter=500)
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# XGBoost
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)


## Compare Model Accuracy

We compare accuracy across all three models to get a quick performance overview.


In [ ]:
# ---------------------------------------------
# Compare model performance
# ---------------------------------------------

from sklearn.metrics import accuracy_score

print("Accuracy Scores:")
print("Logistic Regression:", accuracy_score(y_test, y_pred_lr))
print("Random Forest:", accuracy_score(y_test, y_pred_rf))
print("XGBoost:", accuracy_score(y_test, y_pred_xgb))


Accuracy Scores:
Logistic Regression: 0.8881792276702368
Random Forest: 0.8887687923749632
XGBoost: 0.8892109659035079


##  ROC Curves for All Models

We compute predicted probabilities and plot ROC curves for Logistic Regression,
Random Forest, and XGBoost. This step only works if all models were retrained
after the column name cleaning and train/test split.


In [ ]:
# ---------------------------------------------
#  ROC Curves
# ---------------------------------------------

from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Predicted probabilities
lr_probs = log_reg.predict_proba(X_test)[:, 1]
rf_probs = rf.predict_proba(X_test)[:, 1]
xgb_probs = xgb.predict_proba(X_test)[:, 1]

# ROC values
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_probs)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_probs)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_probs)

# AUC scores
lr_auc = auc(lr_fpr, lr_tpr)
rf_auc = auc(rf_fpr, rf_tpr)
xgb_auc = auc(xgb_fpr, xgb_tpr)

plt.figure(figsize=(8,6))
plt.plot(lr_fpr, lr_tpr, label=f"Logistic Regression (AUC = {lr_auc:.3f})")
plt.plot(rf_fpr, rf_tpr, label=f"Random Forest (AUC = {rf_auc:.3f})")
plt.plot(xgb_fpr, xgb_tpr, label=f"XGBoost (AUC = {xgb_auc:.3f})")

plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for All Models")
plt.legend()
plt.show()


ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- age_(10-20)
- age_(20-30)
- age_(30-40)
- age_(40-50)
- age_(50-60)
- ...
Feature names seen at fit time, yet now missing:
- age_[10-20)
- age_[20-30)
- age_[30-40)
- age_[40-50)
- age_[50-60)
- ...
